# ESA Kelvins Baseline Analysis

Exploratory data analysis and baseline model comparison for satellite conjunction collision risk prediction using the ESA Kelvins space debris dataset.

## 1. Dataset Overview

| Metric | Value |
|---|---|
| Training rows | 162,634 |
| Test rows | 24,484 |
| Original columns | 103 |
| Raw model features | 98 |
| Engineered features | 46 |
| Final ML features | 144 |
| Training events | 10,524 |
| Validation events | 2,630 |
| Event overlap | 0 |

**Target:** `log10(Pc)` — log10 of collision probability, ranging from −30 (negligible) to −1.4 (critical). 41.3% of data sits at the floor value (−30).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

train_df = pd.read_csv("../data/raw/esa_kelvins/train_data.csv")
print(f"Training set: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
train_df.head()

## 2. Target Distribution

The target (`risk` = log10 collision probability) is heavily skewed — over 41% of values are at the floor (−30). This makes it a challenging regression problem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df["risk"], bins=100, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("log₁₀(Collision Probability)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Distribution (Full)")
axes[0].axvline(x=-30, color="red", linestyle="--", label="Floor (−30)")
axes[0].legend()

non_floor = train_df[train_df["risk"] > -30]["risk"]
axes[1].hist(non_floor, bins=80, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("log₁₀(Collision Probability)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Target Distribution (Excluding Floor, n={len(non_floor):,})")

plt.tight_layout()
plt.show()

floor_pct = (train_df["risk"] == -30).mean() * 100
print(f"Floor values (risk = -30): {floor_pct:.1f}%")
print(f"Risk range: [{train_df['risk'].min():.1f}, {train_df['risk'].max():.1f}]")

## 3. Key Feature Distributions

Examining the most physically meaningful features: miss distance, relative speed, and position uncertainty.

In [ ]:
key_features = ["miss_distance", "relative_speed", "time_to_tca"]
available = [f for f in key_features if f in train_df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
if len(available) == 1:
    axes = [axes]

for ax, feat in zip(axes, available):
    ax.hist(train_df[feat].dropna(), bins=80, edgecolor="black", alpha=0.7)
    ax.set_title(feat)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## 4. Missing Values & Data Quality

In [ ]:
null_counts = train_df.isnull().sum()
null_cols = null_counts[null_counts > 0].sort_values(ascending=False)

if len(null_cols) > 0:
    print(f"{len(null_cols)} columns with missing values:\n")
    for col, count in null_cols.items():
        print(f"  {col}: {count:,} ({count / len(train_df) * 100:.1f}%)")
else:
    print("No missing values in any column.")

print(f"\nRows with at least one null: {train_df.isnull().any(axis=1).sum():,}")
print(f"Total rows: {len(train_df):,}")

## 5. Event-Based Splitting

We split by `event_id` (not by row) to prevent temporal leakage — no event appears in both train and validation sets.

In [ ]:
events = train_df["event_id"].unique()
n_events = len(events)
cdms_per_event = train_df.groupby("event_id").size()

print(f"Total unique events: {n_events:,}")
print(f"CDMs per event — mean: {cdms_per_event.mean():.1f}, median: {cdms_per_event.median():.0f}, max: {cdms_per_event.max()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cdms_per_event, bins=50, edgecolor="black", alpha=0.7)
ax.set_xlabel("CDMs per Event")
ax.set_ylabel("Number of Events")
ax.set_title("Distribution of CDM Count per Conjunction Event")
plt.tight_layout()
plt.show()

## 6. Correlation with Target

Top features correlated with collision probability.

In [ ]:
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if "risk" in numeric_cols:
    correlations = train_df[numeric_cols].corr()["risk"].drop("risk").abs().sort_values(ascending=False)
    print("Top 15 features by absolute correlation with risk:\n")
    for feat, corr in correlations.head(15).items():
        print(f"  {feat:40s} {corr:.4f}")

## 7. Model Comparison

Three regression models were trained with event-based 80/20 split on 144 features (98 raw + 46 engineered).

### Results

| Model | Val MAE | Val RMSE | Val R² | Correlation |
|---|---|---|---|---|
| Ridge (baseline) | 5.46 | 6.93 | 0.521 | 0.722 |
| Random Forest | 1.79 | 3.30 | 0.891 | 0.944 |
| **XGBoost (best)** | **1.55** | **2.82** | **0.921** | **0.960** |

XGBoost was selected as the primary model.

In [ ]:
import json

with open("../proofs/pipeline_results.json") as f:
    results = json.load(f)

models = results["model_metrics"]
model_names = list(models.keys())
val_r2 = [models[m]["val"]["r2"] for m in model_names]
val_mae = [models[m]["val"]["mae"] for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#4C78A8", "#F58518", "#E45756"]
axes[0].bar(model_names, val_r2, color=colors, edgecolor="black")
axes[0].set_ylabel("Validation R²")
axes[0].set_title("Model Comparison — R²")
axes[0].set_ylim(0, 1)
for i, v in enumerate(val_r2):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

axes[1].bar(model_names, val_mae, color=colors, edgecolor="black")
axes[1].set_ylabel("Validation MAE")
axes[1].set_title("Model Comparison — MAE (lower is better)")
for i, v in enumerate(val_mae):
    axes[1].text(i, v + 0.1, f"{v:.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 8. Top SHAP Features

SHAP analysis on 200 samples reveals the most important features driving XGBoost's predictions.

In [ ]:
shap_features = results["explainability"]["top_features"]
feat_names = [f["feature"] for f in shap_features]
shap_vals = [f["mean_abs_shap"] for f in shap_features]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(feat_names) - 1, -1, -1)
ax.barh(y_pos, shap_vals, color="#E45756", edgecolor="black")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(feat_names)
ax.set_xlabel("Mean |SHAP| Value")
ax.set_title("Top 10 Feature Importances (SHAP)")

for i, v in enumerate(shap_vals):
    ax.text(v + 0.05, len(feat_names) - 1 - i, f"{v:.2f}", va="center")

plt.tight_layout()
plt.show()

print(f"\nTop feature: {feat_names[0]} (SHAP = {shap_vals[0]:.2f})")
print("This is physically meaningful — miss distance normalized by position uncertainty.")

## 9. High-Risk Detection Performance

Using threshold log10(Pc) ≥ −5.0 for high-risk classification:

| Metric | Value |
|---|---|
| Precision | 0.698 |
| Recall | 0.423 |
| F1 Score | 0.527 |
| ROC-AUC | 0.967 |

The high AUC (0.967) indicates strong discriminative ability. Precision-recall trade-off can be tuned per operational requirements — satellite operators typically prefer higher precision to avoid unnecessary maneuvers.

## 10. Key Takeaways

1. **XGBoost is the clear winner** — R² of 0.921 vs Random Forest's 0.891 and Ridge's 0.521
2. **miss_distance_sigma_ratio** dominates predictions — physically interpretable (miss distance relative to position uncertainty)
3. **Floor values (41.3% at −30)** are the main challenge — the model handles them well but band-level R² suffers
4. **Event-based splitting is critical** — naive row splits would leak temporal information within events
5. **Physics verification provides independent validation** — 20/20 physics checks passed, Akella-Alfriend Pc correlates with ML predictions